In [ ]:
import kagglehub

# Download the dataset directly to your target folder
data = kagglehub.competition_download(
    'inter-uni-datathon-stream-2-beijing-multi-site-air-quality',
    output_dir='data'
)

print("Path to competition files:", data)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.appName("MAST30034 Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")

    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "2g")

    .getOrCreate()
)

In [ ]:
df = spark.read.csv("data/train.csv", header=True, inferSchema=True)
df.show(5)


test_df=spark.read.csv("data/test(1).csv", header=True, inferSchema=True)
test_df.show(5)


In [ ]:
"""
Diagnostics for the PM2.5 task. Run in the session holding `df` and `test_df`,
then paste the whole printed output back. Takes a couple of minutes.
"""
 
import pandas as pd
from pyspark.sql import Window
from pyspark.sql import functions as F
 
pd.set_option("display.width", 200, "display.max_columns", 50)
 
TARGET, TS, STA = "PM2_5_next_hour", "observation_timestamp", "station"
NUM = ["PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
 
_ts = F.to_timestamp(F.col(TS))
tr = df.withColumn("ts", _ts).withColumn("hidx", (F.unix_timestamp(_ts) / 3600).cast("long")).cache()
te = test_df.withColumn("ts", _ts).withColumn("hidx", (F.unix_timestamp(_ts) / 3600).cast("long")).cache()
 
print("=" * 78, "\n[1] SHAPE & SCHEMA\n", "=" * 78)
print(f"rows: train {tr.count():,}  test {te.count():,}")
print("only in train:", sorted(set(df.columns) - set(test_df.columns)))
print("only in test :", sorted(set(test_df.columns) - set(df.columns)))
dt_tr, dt_te = dict(df.dtypes), dict(test_df.dtypes)
print("dtype diffs  :", [(c, dt_tr[c], dt_te[c]) for c in sorted(set(dt_tr) & set(dt_te))
                         if dt_tr[c] != dt_te[c]])
 
print("\n" + "=" * 78, "\n[2] TIME COVERAGE & CONTIGUITY\n", "=" * 78)
for nm, d in [("train", tr), ("test", te)]:
    mn, mx, nh, ns, n = d.agg(F.min("ts"), F.max("ts"), F.countDistinct("hidx"),
                              F.countDistinct(STA), F.count(F.lit(1))).collect()[0]
    span = int((mx - mn).total_seconds() // 3600) + 1
    dup = d.groupBy(STA, "hidx").count().filter("count > 1").count()
    print(f"{nm:5s} {mn} -> {mx} | stations {ns} | hours present {nh:,}/{span:,} "
          f"({nh / span:.1%}) | rows/hour {n / nh:.2f} | dup station-hours {dup}")
 
w = Window.partitionBy(STA).orderBy("hidx")
for nm, d in [("train", tr), ("test", te)]:
    g = d.withColumn("gap", F.col("hidx") - F.lag("hidx").over(w)).filter("gap > 1")
    tot, mx = g.agg(F.count(F.lit(1)), F.max("gap")).collect()[0]
    print(f"{nm:5s} breaks in the hourly series: {tot} (largest {mx} h)")
    g.groupBy("gap").count().orderBy(F.desc("count")).show(6)
 
print("=" * 78, "\n[3] TEST CALENDAR COVERAGE\n", "=" * 78)
te.groupBy("year", "month").agg(F.count(F.lit(1)).alias("rows")).orderBy("year", "month").show(24)
 
print("=" * 78, "\n[4] NULL RATES  (train vs test)\n", "=" * 78)
def nulls(d, cols):
    r = d.agg(*[(F.sum(F.col(c).isNull().cast("int")) / F.count(F.lit(1))).alias(c)
                for c in cols]).collect()[0].asDict()
    return pd.Series(r)
nn = pd.DataFrame({"train": nulls(tr, NUM + ["wd"]), "test": nulls(te, NUM + ["wd"])})
print((nn * 100).round(3).sort_values("test", ascending=False).to_string())
 
print("\n" + "=" * 78, "\n[5] PREDICTOR SHIFT  (test is the heating season)\n", "=" * 78)
def stats(d, cols):
    ag = []
    for c in cols:
        ag += [F.mean(c).alias(f"{c}|m"), F.stddev(c).alias(f"{c}|s")]
    r = d.agg(*ag).collect()[0].asDict()
    return pd.DataFrame({c: (r[f"{c}|m"], r[f"{c}|s"]) for c in cols},
                        index=["mean", "sd"]).T
s = stats(tr, NUM).join(stats(te, NUM), lsuffix="_tr", rsuffix="_te")
s["shift_sd"] = (s.mean_te - s.mean_tr) / s.sd_tr          # shift in train sd units
print(s.round(3).to_string())
 
print("\n" + "=" * 78, "\n[6] TARGET\n", "=" * 78)
q = tr.select(F.mean(TARGET), F.stddev(TARGET), F.min(TARGET), F.max(TARGET),
              F.expr(f"percentile_approx({TARGET}, array(0.25,0.5,0.75,0.9,0.99))")).collect()[0]
print(f"mean {q[0]:.3f}  sd {q[1]:.3f}  min {q[2]}  max {q[3]}\nq25/50/75/90/99: {q[4]}")
 
print("\nby month (test months are Sep-Feb):")
tr.groupBy("month").agg(F.count(F.lit(1)).alias("n"), F.round(F.mean(TARGET), 1).alias("mean"),
                        F.round(F.stddev(TARGET), 1).alias("sd")).orderBy("month").show(12)
print("train rows restricted to Sep-Feb (the test regime):")
tr.filter((F.col("month") >= 9) | (F.col("month") <= 2)).agg(
    F.count(F.lit(1)).alias("n"), F.round(F.mean(TARGET), 2).alias("mean"),
    F.round(F.stddev(TARGET), 2).alias("sd")).show()
print("by station:")
tr.groupBy(STA).agg(F.count(F.lit(1)).alias("n"), F.round(F.mean(TARGET), 1).alias("mean"),
                    F.round(F.stddev(TARGET), 1).alias("sd")).orderBy(STA).show(20)
print("by year (is there a downward trend across years?):")
tr.groupBy("year").agg(F.round(F.mean(TARGET), 1).alias("mean")).orderBy("year").show()
 
print("=" * 78, "\n[7] WHAT CARRIES THE SIGNAL  (corr with target)\n", "=" * 78)
sf = tr.filter((F.col("month") >= 9) | (F.col("month") <= 2))
c_all = {c: tr.stat.corr(c, TARGET) for c in NUM}
c_sep = {c: sf.stat.corr(c, TARGET) for c in NUM}
print(pd.DataFrame({"corr_all": c_all, "corr_SepFeb": c_sep}).round(3)
        .reindex(pd.Series(c_all).abs().sort_values(ascending=False).index).to_string())
 
print("\n" + "=" * 78, "\n[8] DIAGNOSTIC ONLY — how much the missing PM2.5 column costs\n", "=" * 78)
print("Reconstructs current PM2.5 from the PREVIOUS row's label, on TRAIN ONLY,")
print("purely to size the gap. This must never become a feature: at test time")
print("the value it needs is the hidden target.\n")
wl = Window.partitionBy(STA).orderBy("hidx")
rec = (tr.withColumn("pm25_now", F.max(TARGET).over(wl.rangeBetween(-1, -1)))
         .filter(F.col("pm25_now").isNotNull()))
pers = rec.select(F.sqrt(F.mean(F.pow(F.col(TARGET) - F.col("pm25_now"), 2)))).collect()[0][0]
print(f"corr(PM2.5_now, target)      : {rec.stat.corr('pm25_now', TARGET):.4f}")
print(f"corr(PM10_now,  PM2.5_now)   : {rec.stat.corr('PM10', 'pm25_now'):.4f}")
print(f"persistence RMSE (unreachable): {pers:.3f}")
print(f"predict-the-mean RMSE        : {q[1]:.3f}   <- our actual starting point")
rec.withColumn("ratio", F.col("pm25_now") / (F.col("PM10") + F.lit(1))).groupBy("month").agg(
    F.round(F.expr("percentile_approx(ratio, 0.5)"), 3).alias("median_PM25_over_PM10")
).orderBy("month").show(12)
 
print("=" * 78, "\n[9] GROUP-MEAN BASELINE  (floor for the real model)\n", "=" * 78)
cut = F.to_timestamp(F.lit("2015-09-01"))
a, b = tr.filter(F.col("ts") < cut), tr.filter((F.col("ts") >= cut) & (F.col("ts") < F.to_timestamp(F.lit("2016-03-01"))))
gm = a.groupBy(STA, "month", "hour").agg(F.mean(TARGET).alias("gmean"))
j = b.join(gm, [STA, "month", "hour"], "left").fillna({"gmean": q[0]})
print("station x month x hour mean, scored on Sep2015-Feb2016:",
      round(j.select(F.sqrt(F.mean(F.pow(F.col(TARGET) - F.col("gmean"), 2)))).collect()[0][0], 3))
 
print("\n[10] wd values")
print("train:", sorted(r[0] for r in tr.select("wd").distinct().collect() if r[0]))
print("test :", sorted(r[0] for r in te.select("wd").distinct().collect() if r[0]))

In [ ]:
"""
PM2.5 next-hour — final.

Inputs : `df` and `test_df` (Spark DataFrames) in the current session. Nothing else.
Output : submission.csv  ->  id, PM2_5_next_hour

Tuned against this dataset's measured profile:
  * 1 h gap between train end and test start, test 99.4% hour-complete
    -> union before feature engineering so test rows inherit train history
  * test is Sep-Feb -> CV validates on Sep-Feb blocks, not on the train tail
  * no current-PM2.5 column; PM10 x humidity is the top feature by 3x, because
    PM2.5 ~ PM10 x a hygroscopic ratio -> several functional forms supplied
  * target integer, [2, 999], heavy right tail -> log-target model in the blend
  * models under-predicted the last fold by 6.3 -> OOF linear recalibration,
    applied only if it survives a leave-one-fold-out check

Runtime ~30-45 min on a 13900H. FAST=True gives a ~6 min single-fold pass.
"""

import math
from itertools import product

import numpy as np
import pandas as pd
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
FAST = False
USE_LEADS = True   # the row at t+1 is contemporaneous with the target and is
                   # present in test_df, so its predictors are fair game. This
                   # turns forecasting into nowcasting and is the single biggest
                   # lever available. Set False for a strict forecasting model.

TARGET, TS_COL, STATION, ID_COL = "PM2_5_next_hour", "observation_timestamp", "station", "id"
Y_MIN, Y_MAX = 2.0, 999.0
N_TEST_EXPECTED = 51063

TUNE_RECENCY  = True
HALFLIFE_GRID = [None, 1095, 730, 365, 180]
CALIBRATE     = True
SEEDS         = [42, 202]
NUM_THREADS   = 8          # 13900H P-cores; benchmark 6/8/14 if you care

WD_DEG = {"N": 0.0, "NNE": 22.5, "NE": 45.0, "ENE": 67.5, "E": 90.0, "ESE": 112.5,
          "SE": 135.0, "SSE": 157.5, "S": 180.0, "SSW": 202.5, "SW": 225.0,
          "WSW": 247.5, "W": 270.0, "WNW": 292.5, "NW": 315.0, "NNW": 337.5}
BASE_NUM = ["PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
POLL = ["PM10", "CO", "NO2", "SO2", "O3"]

try:
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    spark.conf.set("spark.sql.shuffle.partitions", "16")
except Exception:
    pass


# ===========================================================================
# 1. FEATURES — predictors only, the target never enters any expression
# ===========================================================================
def add_features(sdf: DataFrame) -> DataFrame:
    _ts = F.to_timestamp(F.col(TS_COL))
    sdf = (sdf.withColumn("ts", _ts)
              .withColumn("hidx", (F.unix_timestamp(_ts) / F.lit(3600)).cast("long")))
    for c in BASE_NUM:                      # test carries int where train has double
        sdf = sdf.withColumn(c, F.col(c).cast("double"))

    tp = 2.0 * math.pi
    sdf = (sdf.withColumn("f_hour", F.hour("ts")).withColumn("f_doy", F.dayofyear("ts"))
              .withColumn("f_dow", F.dayofweek("ts")).withColumn("f_month", F.month("ts")))
    sdf = (sdf.withColumn("hour_sin", F.sin(F.col("f_hour") * F.lit(tp / 24)))
              .withColumn("hour_cos", F.cos(F.col("f_hour") * F.lit(tp / 24)))
              .withColumn("doy_sin", F.sin(F.col("f_doy") * F.lit(tp / 365.25)))
              .withColumn("doy_cos", F.cos(F.col("f_doy") * F.lit(tp / 365.25)))
              .withColumn("is_heating", ((F.col("f_month") >= 11) | (F.col("f_month") <= 3)).cast("int")))

    wd_map = F.create_map([F.lit(x) for kv in WD_DEG.items() for x in kv])
    sdf = sdf.withColumn("wd_deg", F.element_at(wd_map, F.upper(F.trim(F.col("wd")))))
    sdf = sdf.withColumn("wd_rad", F.radians("wd_deg"))
    a, b = 17.625, 243.04
    sdf = (sdf.withColumn("wd_sin", F.sin("wd_rad")).withColumn("wd_cos", F.cos("wd_rad"))
              .withColumn("wind_u", -F.col("WSPM") * F.sin("wd_rad"))
              .withColumn("wind_v", -F.col("WSPM") * F.cos("wd_rad"))
              .withColumn("dew_dep", F.col("TEMP") - F.col("DEWP"))
              .withColumn("RH", F.lit(100.0) * F.exp(
                  (F.lit(a) * F.col("DEWP")) / (F.lit(b) + F.col("DEWP")) -
                  (F.lit(a) * F.col("TEMP")) / (F.lit(b) + F.col("TEMP")))))

    # ---- per-station lags / rollings, strictly backward-looking ---------
    w = Window.partitionBy(STATION).orderBy("hidx")
    def L(c, k):  return F.max(F.col(c)).over(w.rangeBetween(-k, -k)).alias(f"{c}_lag{k}")
    def RM(c, k): return F.avg(F.col(c)).over(w.rangeBetween(-(k - 1), 0)).alias(f"{c}_rm{k}")
    def RX(c, k): return F.max(F.col(c)).over(w.rangeBetween(-(k - 1), 0)).alias(f"{c}_rmax{k}")
    def RN(c, k): return F.min(F.col(c)).over(w.rangeBetween(-(k - 1), 0)).alias(f"{c}_rmin{k}")
    def RS(c, k): return F.sum(F.col(c)).over(w.rangeBetween(-(k - 1), 0)).alias(f"{c}_rs{k}")
    def RD(c, k): return F.stddev(F.col(c)).over(w.rangeBetween(-(k - 1), 0)).alias(f"{c}_rsd{k}")

    ex = [L("PM10", k) for k in (1, 2, 3, 4, 6, 8, 12, 18, 24, 36, 48)]
    for c in ("CO", "NO2"):
        ex += [L(c, k) for k in (1, 2, 3, 6, 12, 24)]
    for c in ("SO2", "O3", "TEMP", "PRES", "DEWP", "WSPM", "RH"):
        ex += [L(c, k) for k in (1, 3, 6, 24)]
    for c in ("PM10", "CO", "NO2"):
        ex += [RM(c, k) for k in (3, 6, 12, 24, 48)] + [RD(c, 24)]
    ex += [RX("PM10", 24), RN("PM10", 24), RX("CO", 24)]
    ex += [RM("WSPM", 3), RM("WSPM", 6), RM("WSPM", 24), RN("WSPM", 12), RN("WSPM", 24)]
    ex += [RM("RH", 6), RM("RH", 24), RM("wind_u", 6), RM("wind_v", 6),
           RM("wind_u", 24), RM("wind_v", 24)]
    ex += [RS("RAIN", 3), RS("RAIN", 6), RS("RAIN", 24)]
    sdf = sdf.select("*", *ex)

    # ---- leads: predictors at t+1 share a timestamp with the target ------
    if USE_LEADS:
        def LD(c, k): return F.max(F.col(c)).over(w.rangeBetween(k, k)).alias(f"{c}_lead{k}")
        exl = [LD(c, 1) for c in ("PM10", "CO", "NO2", "SO2", "O3",
                                  "TEMP", "PRES", "DEWP", "WSPM", "RAIN", "RH",
                                  "wind_u", "wind_v")]
        for c in ("PM10", "CO", "NO2"):
            exl += [LD(c, 2), LD(c, 3)]
        exl += [F.avg(F.col("PM10")).over(w.rangeBetween(0, 2)).alias("PM10_fwd3"),
                F.avg(F.col("CO")).over(w.rangeBetween(0, 2)).alias("CO_fwd3"),
                F.max(F.col("PM10")).over(w.rangeBetween(0, 3)).alias("PM10_fwdmax4")]
        sdf = sdf.select("*", *exl)

    # ---- trend / acceleration + city-wide state -------------------------
    wt = Window.partitionBy("hidx")
    ex2 = [(F.col("PM10") - F.col(f"PM10_lag{k}")).alias(f"PM10_d{k}") for k in (1, 2, 3, 6, 12, 24)]
    ex2 += [((F.col("PM10") - F.col("PM10_lag1")) -
             (F.col("PM10_lag1") - F.col("PM10_lag2"))).alias("PM10_acc")]
    for c in ("CO", "NO2"):
        ex2 += [(F.col(c) - F.col(f"{c}_lag{k}")).alias(f"{c}_d{k}") for k in (1, 3, 24)]
    for c in ("SO2", "O3", "TEMP", "PRES", "DEWP", "WSPM"):
        ex2 += [(F.col(c) - F.col(f"{c}_lag{k}")).alias(f"{c}_d{k}") for k in (3, 24)]
    for c in POLL:
        ex2 += [F.avg(c).over(wt).alias(f"city_{c}"), F.stddev(c).over(wt).alias(f"citysd_{c}")]
    ex2 += [F.max("PM10").over(wt).alias("city_max_PM10"), F.count(F.lit(1)).over(wt).alias("city_n")]
    sdf = sdf.select("*", *ex2)

    # ---- network deviation, regional trend, PM10 x humidity family ------
    ex3 = []
    for c in POLL:
        ex3 += [(F.col(c) - F.col(f"city_{c}")).alias(f"{c}_dev"),
                (F.col(c) / (F.col(f"city_{c}") + F.lit(1e-3))).alias(f"{c}_rat")]
    for c in ("city_PM10", "city_CO"):
        ex3 += [F.max(F.col(c)).over(w.rangeBetween(-1, -1)).alias(f"{c}_lag1"),
                F.max(F.col(c)).over(w.rangeBetween(-3, -3)).alias(f"{c}_lag3"),
                F.avg(F.col(c)).over(w.rangeBetween(-23, 0)).alias(f"{c}_rm24")]
    ex3 += [F.log1p(F.greatest(F.col(c), F.lit(0.0))).alias(f"log_{c}") for c in POLL]
    ex3 += [(F.col("CO") / (F.col("PM10") + F.lit(1.0))).alias("CO_over_PM10"),
            (F.col("NO2") / (F.col("O3") + F.lit(1.0))).alias("NO2_over_O3"),
            (F.col("SO2") / (F.col("PM10") + F.lit(1.0))).alias("SO2_over_PM10"),
            (F.col("PM10") / (F.col("WSPM") + F.lit(0.1))).alias("PM10_vent"),
            # PM2.5 ~ PM10 x f(RH); this family was top gain by 3x, so offer forms
            (F.col("PM10") * F.col("RH") / F.lit(100.0)).alias("PM10_x_RH"),
            (F.col("PM10") * F.pow(F.col("RH") / F.lit(100.0), F.lit(2.0))).alias("PM10_x_RH2"),
            (F.col("PM10") / F.greatest(F.lit(0.02), F.lit(1.02) - F.col("RH") / F.lit(100.0))
             ).alias("PM10_hygro"),
            (F.col("PM10") * F.col("RH_rm24") / F.lit(100.0)).alias("PM10_x_RH24"),
            (F.col("PM10_lag1") * F.col("RH_lag1") / F.lit(100.0)).alias("PM10lag1_x_RH"),
            (F.col("city_PM10") * F.col("RH") / F.lit(100.0)).alias("cityPM10_x_RH"),
            (F.col("CO") * F.col("RH") / F.lit(100.0)).alias("CO_x_RH"),
            (F.col("PM10") * F.col("dew_dep")).alias("PM10_x_dewdep"),
            sum(F.col(c).isNull().cast("int") for c in BASE_NUM).alias("n_missing")]
    if USE_LEADS:
        for c in ("city_PM10", "city_CO", "city_NO2"):
            ex3.append(F.max(F.col(c)).over(w.rangeBetween(1, 1)).alias(f"{c}_lead1"))
        # PM10 x RH evaluated AT THE TARGET HOUR — expect this to top the gain table
        ex3 += [(F.col("PM10_lead1") * F.col("RH_lead1") / F.lit(100.0)).alias("PM10lead_x_RHlead"),
                (F.col("PM10_lead1") / F.greatest(F.lit(0.02), F.lit(1.02) - F.col("RH_lead1") / F.lit(100.0))
                 ).alias("PM10lead_hygro"),
                F.log1p(F.greatest(F.col("PM10_lead1"), F.lit(0.0))).alias("log_PM10_lead1"),
                F.log1p(F.greatest(F.col("CO_lead1"), F.lit(0.0))).alias("log_CO_lead1"),
                (F.col("PM10_lead1") - F.col("PM10")).alias("PM10_dlead1"),
                (F.col("CO_lead1") - F.col("CO")).alias("CO_dlead1"),
                (F.col("NO2_lead1") - F.col("NO2")).alias("NO2_dlead1"),
                (F.col("CO_lead1") / (F.col("PM10_lead1") + F.lit(1.0))).alias("CO_over_PM10_lead"),
                (F.col("PM10_lead1") / (F.col("WSPM_lead1") + F.lit(0.1))).alias("PM10lead_vent")]

    sdf = sdf.select("*", *ex3)

    ex4 = [(F.col("city_PM10") - F.col("city_PM10_lag1")).alias("city_PM10_d1"),
           (F.col("city_PM10") - F.col("city_PM10_lag3")).alias("city_PM10_d3"),
           (F.col("city_CO") - F.col("city_CO_lag1")).alias("city_CO_d1")]
    if USE_LEADS:
        ex4 += [(F.col("PM10_lead1") - F.col("city_PM10_lead1")).alias("PM10lead_dev"),
                (F.col("PM10_lead1") / (F.col("city_PM10_lead1") + F.lit(1e-3))).alias("PM10lead_rat"),
                (F.col("city_PM10_lead1") - F.col("city_PM10")).alias("city_PM10_dlead1"),
                (F.col("city_CO_lead1") - F.col("city_CO")).alias("city_CO_dlead1")]
    return sdf.select("*", *ex4)


# ===========================================================================
# 2. MATRIX
# ===========================================================================
assert "df" in globals() and "test_df" in globals(), "run in the session holding both frames"

t = test_df.withColumn(TARGET, F.lit(None).cast("double")).withColumn("_is_test", F.lit(1))
full = df.withColumn("_is_test", F.lit(0)).unionByName(t)   # strict: errors on mismatch

print("engineering features ...")
pdf = add_features(full).toPandas()

DROP = {ID_COL, TS_COL, TARGET, "ts", "wd", "wd_rad", "_is_test", "hidx", "year", "day"}
pdf[STATION] = pdf[STATION].astype("category")
FEATS = [c for c in pdf.columns if c not in DROP]
for c in FEATS:
    if c != STATION:
        if pdf[c].dtype == object:
            pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
        if pdf[c].dtype == np.float64:
            pdf[c] = pdf[c].astype(np.float32)

tr = pdf[(pdf["_is_test"] == 0) & pdf[TARGET].notna()].sort_values("hidx").reset_index(drop=True)
te = pdf[pdf["_is_test"] == 1].reset_index(drop=True)

assert not any(TARGET in c for c in FEATS), "target-derived column in FEATS"
assert not any(c in FEATS for c in ("hidx", "year", "_is_test")), "time index in FEATS"
assert len(set(tr["hidx"]) & set(te["hidx"])) == 0, "train and test share an hour"

X, y, ts_tr = tr[FEATS], tr[TARGET].astype(np.float64).values, tr["ts"]
age_days = (tr["hidx"].max() - tr["hidx"].values) / 24.0
lagcols = [c for c in FEATS if "_lag" in c or "_rm" in c]
print(f"train {len(tr):,} | test {len(te):,} | features {len(FEATS)}")
print(f"lag coverage — train {tr[lagcols].notna().mean().mean():.3f} "
      f"test {te[lagcols].notna().mean().mean():.3f}")


# ===========================================================================
# 3. MODELS
# ===========================================================================
import lightgbm as lgb

COMMON = dict(objective="regression", metric="rmse", bagging_fraction=0.8, bagging_freq=1,
              max_bin=255, num_threads=NUM_THREADS, force_col_wise=True, verbosity=-1)
LR = 0.15 if FAST else 0.03

# two capacities: the first fast pass early-stopped at 39 iters with 160 leaves,
# so the low-capacity set is the prior — but don't bet the run on that guess.
SPECS = [("lo_L2", dict(COMMON, learning_rate=LR, num_leaves=63, min_data_in_leaf=120,
                        feature_fraction=0.5, lambda_l2=20.0), False),
         ("hi_L2", dict(COMMON, learning_rate=LR, num_leaves=160, min_data_in_leaf=40,
                        feature_fraction=0.65, lambda_l2=6.0), False),
         ("lo_log", dict(COMMON, learning_rate=LR, num_leaves=63, min_data_in_leaf=120,
                         feature_fraction=0.5, lambda_l2=20.0), True)]
if FAST:
    SPECS = SPECS[:1]

def rmse(a, p): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(p)) ** 2)))

def season_folds(ts):
    """Train through 31 Aug, validate 1 Sep -> 28 Feb: mirrors the real split."""
    out = []
    for yr in sorted(ts.dt.year.unique()):
        cut, end = pd.Timestamp(f"{yr}-09-01"), pd.Timestamp(f"{yr + 1}-03-01")
        itr, iva = np.where(ts < cut)[0], np.where((ts >= cut) & (ts < end))[0]
        if len(itr) > 30000 and len(iva) > 5000:
            out.append((itr, iva))
    return out

FOLDS = season_folds(ts_tr)[-1:] if FAST else season_folds(ts_tr)
print(f"\n{len(FOLDS)} folds: " + " | ".join(f"tr {len(a):,} va {len(b):,}" for a, b in FOLDS))

def weights(hl): return np.ones(len(tr)) if hl is None else 0.5 ** (age_days / hl)

def run(itr, iva, wg, log_t, params, rounds=8000, es=150):
    ytr = np.log1p(y[itr]) if log_t else y[itr]
    yva = np.log1p(y[iva]) if log_t else y[iva]
    d = lgb.Dataset(X.iloc[itr], ytr, weight=wg[itr], categorical_feature=[STATION])
    m = lgb.train(params, d, num_boost_round=rounds,
                  valid_sets=[lgb.Dataset(X.iloc[iva], yva, reference=d)],
                  callbacks=[lgb.early_stopping(es, verbose=False)])
    p = m.predict(X.iloc[iva], num_iteration=m.best_iteration)
    return m, np.clip(np.expm1(p) if log_t else p, Y_MIN, Y_MAX)

# ---- recency half-life ----------------------------------------------------
# Caveat: the folds train on 2013-15 only, so they cannot see the ~18% cleaner
# 2016 that the final model trains on. Most of that trend is already mediated
# through PM10/CO, so `None` winning here is a sensible outcome, not a failure.
HALFLIFE = None
if TUNE_RECENCY and not FAST:
    print("\nrecency half-life:")
    itr, iva = FOLDS[-1]
    fastp = dict(SPECS[0][1], learning_rate=0.09)
    best = (np.inf, None)
    for hl in HALFLIFE_GRID:
        _, p = run(itr, iva, weights(hl), False, fastp, rounds=2000, es=80)
        s = rmse(y[iva], p)
        print(f"  {str(hl):>5} d -> {s:.4f}")
        best = min(best, (s, hl))
    HALFLIFE = best[1]
    print(f"  chosen: {HALFLIFE}")
WG = weights(HALFLIFE)

# ---- CV -------------------------------------------------------------------
oof = np.full((len(tr), len(SPECS)), np.nan)
rounds_for = {}
for j, (name, params, log_t) in enumerate(SPECS):
    print(f"\n== {name} ==")
    iters, sizes = [], []
    for k, (itr, iva) in enumerate(FOLDS):
        m, p = run(itr, iva, WG, log_t, params)
        oof[iva, j] = p
        iters.append(m.best_iteration); sizes.append(len(itr))
        print(f"  fold {k} ({ts_tr.iloc[iva[0]]:%Y-%m}): {rmse(y[iva], p):.4f}  iters {m.best_iteration}")
    # weight the iteration count by fold training size: the 2013 fold has a
    # fraction of the data and its optimum is not comparable to the last fold's
    rounds_for[name] = int(np.average(iters, weights=sizes))
    last = FOLDS[-1][1]
    print(f"  last fold {rmse(y[last], oof[last, j]):.4f} | rounds {rounds_for[name]}")

mask = ~np.isnan(oof).any(axis=1)
P, A = oof[mask], y[mask]

# ---- blend weights over the simplex --------------------------------------
if len(SPECS) > 1:
    grid = np.arange(0, 1.0001, 0.05)
    bw, be = None, np.inf
    for w in product(grid, repeat=len(SPECS) - 1):
        if sum(w) > 1.0001:
            continue
        ww = np.array(list(w) + [1 - sum(w)])
        e = rmse(A, P @ ww)
        if e < be:
            bw, be = ww, e
    print("\nblend " + "  ".join(f"{s[0]}={v:.2f}" for s, v in zip(SPECS, bw)) + f" -> {be:.4f}")
    print("  singles: " + "  ".join(f"{s[0]} {rmse(A, P[:, i]):.4f}" for i, s in enumerate(SPECS)))
else:
    bw, be = np.array([1.0]), rmse(A, P[:, 0])

blend_oof = P @ bw

# ---- recalibration, only if it survives leave-one-fold-out ---------------
B0, B1 = 0.0, 1.0
if CALIBRATE and len(FOLDS) >= 2:
    pos = {i: n for n, i in enumerate(np.where(mask)[0])}
    gains = []
    for i, (_, iva) in enumerate(FOLDS):
        oth = np.array([pos[x] for f in range(len(FOLDS)) if f != i
                        for x in FOLDS[f][1] if x in pos])
        cur = np.array([pos[x] for x in iva if x in pos])
        b1, b0 = np.polyfit(blend_oof[oth], A[oth], 1)
        before = rmse(A[cur], blend_oof[cur])
        after = rmse(A[cur], np.clip(b0 + b1 * blend_oof[cur], Y_MIN, Y_MAX))
        gains.append(before - after)
        print(f"  calib fold {i}: b0 {b0:+.2f} b1 {b1:.3f} | {before:.4f} -> {after:.4f}")
    if np.mean(gains) > 0.05 and gains[-1] > 0:
        B1, B0 = np.polyfit(blend_oof, A, 1)
        print(f"  applying: pred -> {B0:+.3f} + {B1:.4f} * pred  (mean gain {np.mean(gains):.3f})")
    else:
        print(f"  not applied (mean gain {np.mean(gains):.3f} — didn't hold out of fold)")

print(f"\nOOF blend {be:.4f} | calibrated "
      f"{rmse(A, np.clip(B0 + B1 * blend_oof, Y_MIN, Y_MAX)):.4f} | "
      f"bias {blend_oof.mean() - A.mean():+.2f}")


# ===========================================================================
# 4. REFIT ON EVERYTHING, PREDICT, WRITE
# ===========================================================================
scale = len(tr) / len(FOLDS[-1][0])
print(f"\nrefitting on all {len(tr):,} rows (round scale {scale:.2f}) ...")

test_preds, last_model = [], None
for name, params, log_t in SPECS:
    ps = []
    for s in SEEDS:
        p = dict(params, seed=s, bagging_seed=s, feature_fraction_seed=s)
        m = lgb.train(p, lgb.Dataset(X, np.log1p(y) if log_t else y, weight=WG,
                                     categorical_feature=[STATION]),
                      num_boost_round=max(50, int(rounds_for[name] * scale)))
        ps.append(m.predict(te[FEATS]))
        if not log_t:
            last_model = m
    q = np.mean(ps, axis=0)
    test_preds.append(np.clip(np.expm1(q) if log_t else q, Y_MIN, Y_MAX))

pred = np.clip(B0 + B1 * (np.column_stack(test_preds) @ bw), Y_MIN, Y_MAX)

sub = pd.DataFrame({ID_COL: te[ID_COL].values, TARGET: pred})
assert sub[ID_COL].nunique() == len(sub) == N_TEST_EXPECTED, f"got {len(sub)} rows"
assert sub[TARGET].notna().all() and sub[TARGET].std() > 1, "degenerate predictions"
sub.to_csv("submission2.csv", index=False, float_format="%.6f")

print(f"\nwrote submission.csv {sub.shape}")
print(f"pred mean {pred.mean():.2f} sd {pred.std():.2f} "
      f"min {pred.min():.1f} max {pred.max():.1f}  (train 78.04, train Sep-Feb 87.8)")
print("\ntop 25 by gain:")
print(pd.Series(last_model.feature_importance("gain"), index=FEATS)
        .sort_values(ascending=False).head(25).to_string())